# Africa Flight Price Fine-tuning

Fine-tune **GPT-4o mini** (a different model from the course's GPT-4.1-nano) on **flight prices in the Africa continent** across multiple countries.

- **Data**: Loaded from **Hugging Face** dataset `Karosi/africa-flight-prices` (or local `data/flight_prices_africa.csv` if not yet uploaded).
- **Base model**: `gpt-4o-mini` (OpenAI fine-tunable).
- **Task**: Answer questions like "What is the flight price from Lagos, Nigeria to Nairobi, Kenya?" with the price, e.g. "$420.00".

In [ ]:
import os
import json
import random
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset

In [ ]:
# Load .env (tries cwd, repo root, and this notebook's folder)
load_dotenv(override=True)
for _path in [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "week6" / "community-contributions" / "adeyemi-kayode" / ".env",
]:
    if _path.exists():
        load_dotenv(_path, override=True)

# Fine-tuning requires the OFFICIAL OpenAI API
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is required for fine-tuning. OpenRouter cannot be used for this step. "
        "Add OPENAI_API_KEY to .env (get it from https://platform.openai.com/api-keys)."
    )
client = OpenAI(api_key=api_key)  # Uses api.openai.com (default)
print("OpenAI client ready for fine-tuning (using OPENAI_API_KEY from env)")

## Load Africa flight price data from Hugging Face

We use the **Hugging Face dataset** [Karosi/africa-flight-prices](https://huggingface.co/datasets/Karosi/africa-flight-prices). If it is not on the Hub yet, we fall back to the local CSV (run the "Upload dataset to Hugging Face" cell below once to publish it).

In [ ]:
# Hugging Face dataset ID for Africa flight prices (upload once with the cell below if needed)
HF_DATASET_ID = "Karosi/africa-flight-prices"

def load_africa_flight_data():
    """Load from Hugging Face Hub if available, else from local CSV."""
    try:
        ds = load_dataset(HF_DATASET_ID, split="train")
        df = ds.to_pandas()
        print(f"Loaded {len(df)} flight price examples from Hugging Face: {HF_DATASET_ID}")
    except Exception as e:
        print(f"Hugging Face dataset not found ({e}). Using local CSV.")
        data_path = Path("data/flight_prices_africa.csv")
        df = pd.read_csv(data_path)
        print(f"Loaded {len(df)} flight price examples from {data_path}")
    df["price_usd"] = pd.to_numeric(df["price_usd"], errors="coerce").astype(float)
    return df

df = load_africa_flight_data()
df.head(10)

### (Optional) Upload dataset to Hugging Face (run once)

Run this cell once to publish the Africa flight price data to the Hub. You need `HF_TOKEN` in `.env` or `huggingface-cli login`. After uploading, anyone can load it with `load_dataset("Karosi/africa-flight-prices")`.

In [ ]:
# Uncomment and run once to upload this dataset to Hugging Face:
# from datasets import Dataset
# df_upload = pd.read_csv(Path("data/flight_prices_africa.csv"))
# df_upload["price_usd"] = pd.to_numeric(df_upload["price_usd"], errors="coerce")
# dataset = Dataset.from_pandas(df_upload)
# dataset.push_to_hub(HF_DATASET_ID)
# print(f"Uploaded to https://huggingface.co/datasets/{HF_DATASET_ID}")

## Train / validation split (OpenAI recommends 50–100+ examples; we use ~80 train, ~15 val)

In [ ]:
random.seed(42)
rows = df.to_dict("records")
random.shuffle(rows)
n_val = max(15, len(rows) // 6)
train_rows = rows[:-n_val]
val_rows = rows[-n_val:]
print(f"Train: {len(train_rows)}, Validation: {len(val_rows)}")

## Build messages and JSONL for fine-tuning

In [ ]:
def messages_for(row):
    """One training example: user asks for flight price, assistant replies with price."""
    prompt = (
        f"What is the flight price from {row['origin_city']}, {row['origin_country']} "
        f"to {row['destination_city']}, {row['destination_country']}? "
        f"Respond with the price in USD only, no explanation."
    )
    price = row["price_usd"]
    answer = f"${price:.2f}"
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer}
    ]

In [ ]:
def make_jsonl(rows):
    result = ""
    for row in rows:
        messages = messages_for(row)
        result += json.dumps({"messages": messages}) + "\n"
    return result.strip()

def write_jsonl(rows, filename):
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    with open(filename, "w") as f:
        f.write(make_jsonl(rows))

In [ ]:
write_jsonl(train_rows, "jsonl/fine_tune_train.jsonl")
write_jsonl(val_rows, "jsonl/fine_tune_validation.jsonl")
print("Wrote jsonl/fine_tune_train.jsonl and jsonl/fine_tune_validation.jsonl")
print("Sample:", make_jsonl(train_rows[:1]))

## Upload files and create fine-tuning job (model: gpt-4o-mini)

In [ ]:
# Use path that works from notebook dir; pass filename so OpenAI gets the .jsonl suffix
train_path = Path("jsonl/fine_tune_train.jsonl").resolve()
if not train_path.exists():
    raise FileNotFoundError(f"Not found: {train_path}. Run cells above and run from the notebook folder.")
with open(train_path, "rb") as f:
    train_file = client.files.create(file=("fine_tune_train.jsonl", f), purpose="fine-tune")
print("Train file id:", train_file.id)

In [ ]:
val_path = Path("jsonl/fine_tune_validation.jsonl").resolve()
with open(val_path, "rb") as f:
    validation_file = client.files.create(file=("fine_tune_validation.jsonl", f), purpose="fine-tune")
print("Validation file id:", validation_file.id)

In [ ]:
# Use gpt-4o-mini (different from the course's gpt-4.1-nano)
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="africa-flight-price"
)
job_id = job.id
print("Fine-tuning job id:", job_id)

In [ ]:
# Check status (run this cell until status is 'succeeded')
status = client.fine_tuning.jobs.retrieve(job_id)
print("Status:", status.status)
print("Fine-tuned model:", getattr(status, "fine_tuned_model", None))
status

In [ ]:
# Optional: list recent events
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

## Test the fine-tuned model

In [ ]:
# After job status is 'succeeded', get the model name
job_info = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_name = job_info.fine_tuned_model
print("Model:", fine_tuned_model_name)

In [ ]:
def ask_flight_price(origin_city, origin_country, destination_city, destination_country):
    prompt = (
        f"What is the flight price from {origin_city}, {origin_country} "
        f"to {destination_city}, {destination_country}? "
        f"Respond with the price in USD only, no explanation."
    )
    response = client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=15
    )
    return response.choices[0].message.content

In [ ]:
# Example: Lagos to Nairobi (expected around $420)
print(ask_flight_price("Lagos", "Nigeria", "Nairobi", "Kenya"))
# Example: Johannesburg to Harare (expected around $205)
print(ask_flight_price("Johannesburg", "South Africa", "Harare", "Zimbabwe"))

# Lagos to Nairobi (expected around $420)
Result: $580.00

# Johannesburg to Harare (expected around $205)
Result: $105.00
